In [1]:
import scanpy as sc
import liana

/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(ms

## Load data

In [62]:
slide = "Visium_18"
pseudocell_spatial = sc.read_h5ad(f"../results/cardiomyocyte_subtypes/cell2location_map/spot_celltype_pseudobulks_{slide}.h5ad")
pseudocell_spatial.var["feature_name"] = pseudocell_spatial.var_names
pseudocell_spatial.var_names = pseudocell_spatial.var["SYMBOL"]

In [63]:
pseudocell_spatial

AnnData object with n_obs × n_vars = 25004 × 11613
    obs: 'spot', 'cell_type', 'molecular_niche', 'celltype_niche'
    var: 'feature_types', 'genome', 'SYMBOL', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'rps', 'mrp', 'rpl', 'duplicated', 'feature_name'
    obsm: 'spatial'

In [64]:
pseudocell_spatial.var.head(2)

,feature_types,genome,SYMBOL,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,mt,rps,mrp,rpl,duplicated,feature_name
SYMBOL,,,,,,,,,,,,,,,
DPM1,Gene Expression,GRCh38,DPM1,931,0.314804,0.273688,73.994413,1127.0,7.028202,False,False,False,False,False,ENSG00000000419
SCYL3,Gene Expression,GRCh38,SCYL3,256,0.075698,0.072970,92.849162,271.0,5.605802,False,False,False,False,False,ENSG00000000457


# Infer cell communication for both niche 1 and niche 2

### Niche 1

In [65]:
# Cut by molecular niches
rna_niche_1 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_1", :]

liana.method.cellphonedb(
    rna_niche_1, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


4 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.53 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 12348 samples and 11609 features


100%|██████████| 1000/1000 [00:23<00:00, 42.45it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


### Format network for ReCoN

In [66]:
ccc_layer_niche_1 = rna_niche_1.uns["cpdb_res"].copy()
ccc_layer_niche_1 = ccc_layer_niche_1[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_1.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_1["celltype_source"] = rna_niche_1.uns["cpdb_res"]["source"]
ccc_layer_niche_1["celltype_target"] = rna_niche_1.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_1 = ccc_layer_niche_1.loc[~ccc_layer_niche_1.isna().sum(1).astype(bool), :]#.shape

### Save network

In [67]:
ccc_layer_niche_1.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_1_{slide}.csv")

### Niche 5

### Format network for ReCoN

In [68]:
# Cut by molecular niches
rna_niche_5 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_5", :]

liana.method.cellphonedb(
    rna_niche_5, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


201 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.54 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 3297 samples and 11412 features


100%|██████████| 1000/1000 [00:06<00:00, 163.78it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [69]:
ccc_layer_niche_5 = rna_niche_5.uns["cpdb_res"].copy()
ccc_layer_niche_5 = ccc_layer_niche_5[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_5.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_5["celltype_source"] = rna_niche_5.uns["cpdb_res"]["source"]
ccc_layer_niche_5["celltype_target"] = rna_niche_5.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_5 = ccc_layer_niche_5.loc[~ccc_layer_niche_5.isna().sum(1).astype(bool), :]#.shape

### Save network

In [70]:
ccc_layer_niche_5.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_5_{slide}.csv")

### Niche 2

In [71]:
# Cut by molecular niches
rna_niche_2 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_2", :]

liana.method.cellphonedb(
    rna_niche_2, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format
2356 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.63 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 168 samples and 9257 features


100%|██████████| 1000/1000 [00:01<00:00, 558.66it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [72]:
ccc_layer_niche_2 = rna_niche_2.uns["cpdb_res"].copy()
ccc_layer_niche_2 = ccc_layer_niche_2[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_2.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_2["celltype_source"] = rna_niche_2.uns["cpdb_res"]["source"]
ccc_layer_niche_2["celltype_target"] = rna_niche_2.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_2 = ccc_layer_niche_2.loc[~ccc_layer_niche_2.isna().sum(1).astype(bool), :]#.shape

### Save network

In [73]:
ccc_layer_niche_5.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_2_{slide}.csv")

### Niche 4

In [74]:
# Cut by molecular niches
rna_niche_4 = pseudocell_spatial[pseudocell_spatial.obs["celltype_niche"] == "ctniche_4", :]
liana.method.cellphonedb(
    rna_niche_4, 
    expr_prop=0.0,
    resource_name="consensus",
    use_raw=False,
    groupby="cell_type",
    verbose=True,
    key_added='cpdb_res'
)

Using `.X`!
Converting mat to CSR format


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/anndata/_core/anndata.py:401: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


72 features of mat are empty, they will be removed.


/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/pandas/core/indexing.py:1857: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:150: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


0.53 of entities in the resource are missing from the data.
Generating ligand-receptor stats for 4788 samples and 11541 features


100%|██████████| 1000/1000 [00:07<00:00, 132.85it/s]
/pasteur/appa/homes/rtrimbou/miniconda3/envs/snakemake/envs/recon/lib/python3.12/site-packages/liana/method/_Method.py:265: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


### Format network for ReCoN

In [75]:
ccc_layer_niche_4 = rna_niche_4.uns["cpdb_res"].copy()
ccc_layer_niche_4 = ccc_layer_niche_4[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_layer_niche_4.rename(
    columns={
        "lr_means":"weight",
        "ligand":"source",
        "receptor":"target",
        "source":"celltype_source",
        "target":"celltype_target"},
    inplace=True)
ccc_layer_niche_4["celltype_source"] = rna_niche_4.uns["cpdb_res"]["source"]
ccc_layer_niche_4["celltype_target"] = rna_niche_4.uns["cpdb_res"]["target"]

# Keep only positive connections
#ccc_layer = ccc_layer[ccc_layer["weight"]>ccc_layer.weight.quantile(0.3)]
ccc_layer_niche_4 = ccc_layer_niche_4.loc[~ccc_layer_niche_4.isna().sum(1).astype(bool), :]#.shape

### Save network

In [76]:
ccc_layer_niche_4.to_csv(f"../results/cardiomyocyte_subtypes/cell_communication_niche_4_{slide}.csv")